# nb_00_config — shared configuration, DQ rules and lake helpers

Referenced by the three layer notebooks via `%run nb_00_config`. Defines **functions and
constants only** (no side effects, no parameters of its own), so it is safe to include from
any session. Kept in exact parity with `pipeline/config.py` + `pipeline/dq_rules.py` in this
repository. DQ vocabulary: **DQ-Q1** `INVALID_COORDINATES` (quarantine, hard) and
**DQ-W1** `MISSING_OR_UNKNOWN_CITY` (warning, soft — row still publishes).

In [ ]:
"""Shared constants, config factory, DQ predicates, and bronze-run helpers."""
import json
import logging
from datetime import datetime, timezone
from types import SimpleNamespace

import requests
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

try:  # Synapse runtime provides notebookutils; the local test harness injects a shim
    from notebookutils import mssparkutils  # type: ignore
except ImportError:
    mssparkutils  # noqa: B018

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("university_chapters")

FEATURE_SERVER_URL = (
    "https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/"
    "UniversityChapters_Public/FeatureServer/0"
)
QUERY_URL = f"{FEATURE_SERVER_URL}/query"
WHERE_CLAUSE = "State IN ('CA','OR','WA')"
IN_SCOPE_STATES = ("CA", "OR", "WA")
PAGE_SIZE = 1000
HTTP_TIMEOUT_SECONDS = 30

REASON_INVALID_COORDINATES = "INVALID_COORDINATES"
REASON_MISSING_OR_UNKNOWN_CITY = "MISSING_OR_UNKNOWN_CITY"
DQ_STATUS_OK, DQ_STATUS_WARNING = "OK", "WARNING"
LON_MIN, LON_MAX, LAT_MIN, LAT_MAX = -180.0, 180.0, -90.0, 90.0

GOLD_COLUMNS = [
    "chapter_id", "chapter_name", "city", "state", "longitude", "latitude",
    "dq_status", "dq_warnings", "ingest_run_id", "ingested_at_utc",
]


class PipelineError(RuntimeError):
    """Any raise fails the notebook activity -> the pipeline run fails loudly."""


def init_config(storage_account, lake_container="lake"):
    """All lake paths, derived once from the two identity parameters."""
    if not storage_account:
        raise PipelineError("Parameter 'storage_account' is required (ADLS Gen2 account name).")
    lake = f"abfss://{lake_container}@{storage_account}.dfs.core.windows.net"
    d = "university_chapters"
    return SimpleNamespace(
        lake=lake,
        bronze_root=f"{lake}/bronze/{d}",
        silver_path=f"{lake}/silver/{d}",
        gold_path=f"{lake}/gold/{d}/v1",
        quarantine_path=f"{lake}/quarantine/{d}",
        runs_path=f"{lake}/_runs/{d}",
        fixture_path=f"{lake}/fixtures/university_chapters_fixture.json",
    )


def new_run_id(now=None):
    now = now or datetime.now(timezone.utc)
    return now.strftime("run_%Y%m%dT%H%M%SZ")


# ---------------------------------------------------------------- DQ rules --
def invalid_coordinates(lon, lat):
    """DQ-Q1: missing, null, non-numeric (null after try_cast), NaN or out-of-range."""
    lon_bad = lon.isNull() | F.isnan(lon) | (lon < F.lit(LON_MIN)) | (lon > F.lit(LON_MAX))
    lat_bad = lat.isNull() | F.isnan(lat) | (lat < F.lit(LAT_MIN)) | (lat > F.lit(LAT_MAX))
    return lon_bad | lat_bad


def missing_or_unknown_city(city):
    """DQ-W1: null, blank/whitespace, or literal 'UNKNOWN' (case-insensitive)."""
    trimmed = F.trim(city)
    return city.isNull() | (trimmed == F.lit("")) | (F.upper(trimmed) == F.lit("UNKNOWN"))


def with_dq_flags(df):
    quarantined = invalid_coordinates(F.col("longitude"), F.col("latitude"))
    warned = missing_or_unknown_city(F.col("city"))
    return (
        df.withColumn("is_quarantined", quarantined)
        .withColumn("quarantine_reason", F.when(quarantined, F.lit(REASON_INVALID_COORDINATES)))
        .withColumn("dq_status",
                    F.when(warned, F.lit(DQ_STATUS_WARNING)).otherwise(F.lit(DQ_STATUS_OK)))
        .withColumn("dq_warnings",
                    F.when(warned, F.array(F.lit(REASON_MISSING_OR_UNKNOWN_CITY)))
                    .otherwise(F.array().cast("array<string>")))
    )


# ------------------------------------------------------- bronze-run helpers --
def find_bronze_run_dir(cfg, run_id):
    """Locate a bronze run folder by run_id across ingest_date=... partitions."""
    try:
        date_dirs = mssparkutils.fs.ls(cfg.bronze_root)
    except Exception as e:
        raise PipelineError(f"No bronze root at {cfg.bronze_root}: {e}")
    for dd in sorted(date_dirs, key=lambda f: f.name, reverse=True):
        for rd in mssparkutils.fs.ls(dd.path):
            if rd.name.rstrip("/") == f"run_id={run_id}":
                return rd.path
    raise PipelineError(f"No bronze folder found for run_id={run_id} under {cfg.bronze_root}")


def read_ingest_metadata(bronze_dir):
    # Read via fs utilities, NOT spark.read: Hadoop readers silently filter
    # underscore-prefixed files (the very convention that keeps this sidecar
    # out of the page_*.json data reads).
    try:
        content = mssparkutils.fs.head(f"{bronze_dir}/_ingest_metadata.json", 65536)
    except Exception as e:
        raise PipelineError(f"Missing _ingest_metadata.json in {bronze_dir}: {e}")
    return json.loads(content)